# Centrality Analysis — Hypothesis 1

Computes Degree, Closeness, Betweenness, and Eccentricity on the largest connected component and compares structural positions across stakeholder groups.


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 1. Load the weighted edge list and classified node table
edges = pd.read_csv("edges.csv")  # Source, Target, Weight
nodes = pd.read_csv("nodes_classification.csv")  # Id, Label, stakeholder_type

# 2. Build the undirected weighted graph
G = nx.Graph()
for _, row in edges.iterrows():
    G.add_edge(row["Source"], row["Target"], weight=row["Weight"])

# 3. Restrict the analysis to the largest connected component
largest_cc = max(nx.connected_components(G), key=len)
G_sub = G.subgraph(largest_cc).copy()

# 4. Compute centrality measures
degree_centrality = nx.degree_centrality(G_sub)
closeness_centrality = nx.closeness_centrality(G_sub)
betweenness_centrality = nx.betweenness_centrality(G_sub, normalized=True)
eccentricity = nx.eccentricity(G_sub)

# 5. Create the centrality table
centrality_df = pd.DataFrame({
    "Id": list(G_sub.nodes()),
    "Degree": [degree_centrality[node] for node in G_sub.nodes()],
    "Closeness": [closeness_centrality[node] for node in G_sub.nodes()],
    "Betweenness": [betweenness_centrality[node] for node in G_sub.nodes()],
    "Eccentricity": [eccentricity[node] for node in G_sub.nodes()]
})

# 6. Merge centrality scores with stakeholder metadata
merged = pd.merge(nodes, centrality_df, on="Id")

# 7. Overall network summary statistics
print("\nOverall network centrality summary")
print(merged[["Degree", "Closeness", "Betweenness", "Eccentricity"]].describe())

# 8. Compare citizen organizations and local governments with the full network
target_subset = merged[
    merged["stakeholder_type"].isin(["Citizen Organization", "Local Government"])
]
print("\nCitizen organizations and local governments: centrality summary")
print(target_subset[["Degree", "Closeness", "Betweenness", "Eccentricity"]].describe())

# 9. Mean centrality by stakeholder type
group_stats = merged.groupby("stakeholder_type")[[
    "Degree", "Closeness", "Betweenness", "Eccentricity"
]].mean()
print("\nMean centrality by stakeholder type")
print(group_stats)


In [ ]:
import matplotlib.pyplot as plt

centrality_columns = ["Degree", "Closeness", "Betweenness"]

# Histograms
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5))
for i, column in enumerate(centrality_columns):
    merged[column].plot(kind="hist", ax=axes[i], bins=30, edgecolor="white")
    axes[i].set_title(f"Histogram of {column}")
    axes[i].set_xlabel(f"{column} Centrality")
    axes[i].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

# Box plots
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5))
for i, column in enumerate(centrality_columns):
    axes[i].boxplot(merged[column].dropna())
    axes[i].set_title(f"Box Plot of {column}")
    axes[i].set_ylabel(f"{column} Centrality")
plt.tight_layout()
plt.show()


## Centrality Analysis Summary

### 1. Degree Centrality
- Most nodes have very low degree-centrality values, indicating that direct connections are concentrated among a relatively small set of organizations.
- The distribution is consistent with a hub-dominated network rather than uniformly distributed connectivity.

### 2. Closeness Centrality
- Closeness captures how near an organization is to other nodes through shortest paths.
- Higher-scoring organizations occupy positions with comparatively efficient access to the rest of the connected network.

### 3. Betweenness Centrality
- Most nodes have values close to zero, while a small number of organizations account for a disproportionate share of shortest-path brokerage.
- These nodes are potential structural bridges or bottlenecks in the observed co-occurrence network.

### Overall Interpretation
- Centrality is unevenly distributed across the smart-city network.
- Citizen organizations and local governments are comparatively peripheral in the aggregate analysis, while a small set of organizations occupies structurally influential positions.
- These measures describe **network position in news-based co-occurrence data**; they should not be interpreted as direct evidence of formal authority or actual information flow.
